# Project 2 — Fraud Detection Pipeline
### DecodeLabs Data Science Industrial Training Kit · Batch 2026

**Goal:** build a leak-free classification pipeline to flag potentially fraudulent
orders using imbalanced-class techniques (SMOTE), comparing a linear baseline
(Logistic Regression) against an ensemble model (Random Forest), evaluated with
Precision, Recall, F1, and ROC-AUC — **not accuracy**, which is misleading on
imbalanced data.

**Input:** this project reuses the cleaned, feature-engineered output of Project 1
(`data/raw/project1_cleaned_dataset.csv`) rather than re-cleaning raw data.

## ⚠️ Important caveat: `IsFraud` is a proxy label, not real fraud data

The Project 1 orders dataset has no genuine fraud/legitimate column. To still
practice a real fraud-detection *pipeline* on it, orders with `OrderStatus` of
**"Returned"** or **"Cancelled"** are treated as a stand-in for "fraudulent" —
a common technique when true labels aren't available, since problem orders often
correlate with disputes or bad-actor behavior in real businesses.

**This is explicitly NOT a claim that these orders are confirmed fraud.** Every
result below describes *predicting problem orders*, and the modeling machinery
(leak-free splits, SMOTE, GridSearchCV, Precision/Recall/ROC-AUC evaluation) is
exactly what a real fraud pipeline would use — the labels are the only
simplification here.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from src.data_loader import load_project1_dataset
from src.target_builder import build_fraud_target, class_balance_report
from src.feature_engineering import engineer_fraud_features, build_feature_matrix, drop_leaky_columns
from src.modeling import (
    build_logistic_regression_pipeline, build_random_forest_pipeline,
    LOGISTIC_REGRESSION_PARAM_GRID, RANDOM_FOREST_PARAM_GRID, tune_model, RANDOM_STATE,
)
from src.evaluation import evaluate_model, results_to_dataframe
from src.plotting import plot_confusion_matrix, plot_roc_curves, plot_precision_recall_curves, plot_feature_importance, plot_metric_comparison

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 40)

RAW_PATH = "../data/raw/project1_cleaned_dataset.csv"
TEST_SIZE = 0.2

## 1. Load Project 1's Output & Build the Proxy Target

In [2]:
df_raw = load_project1_dataset(RAW_PATH)
print(f"Shape: {df_raw.shape}")

df_labeled = build_fraud_target(df_raw)
balance = class_balance_report(df_labeled)
balance

Shape: (1200, 25)


,label,count,pct
0,Legitimate (0),703,58.58
1,Fraud-proxy (1),497,41.42


In [3]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(balance["label"], balance["count"], color=["#4C72B0", "#C44E52"])
for i, v in enumerate(balance["count"]):
    ax.text(i, v + 5, f"{v}\n({balance['pct'][i]}%)", ha="center")
ax.set_title("Class Balance — IsFraud (proxy target)")
ax.set_ylabel("Order count")
plt.tight_layout()
plt.show()

**Observation:** the proxy target lands at roughly 41% "fraud" / 59% "legitimate"
— nowhere near the extreme (<1%) imbalance real fraud datasets have. This is a direct
consequence of the synthetic data generator giving each of the 5 order statuses
roughly equal probability (~20% each), so `Returned + Cancelled ≈ 40%`. The
techniques in this notebook (SMOTE, recall-first tuning, leak-free splits) are still
applied exactly as a genuinely imbalanced problem would require — but it's worth
being upfront that this dataset's imbalance is much milder than a real fraud
scenario, which will show up later as more modest lift from SMOTE than you'd see on
a truly rare-event problem.

## 2. Feature Engineering

New features engineered specifically for this fraud-detection track:

| Feature | Formula | Purpose |
|---|---|---|
| `AvgItemValue` | `TotalPrice / ItemsInCart` | Average value per item sitting in the cart |
| `ItemsPerOrder` | `ItemsInCart / (Quantity + 1)` | Cart-to-purchase ratio |
| `IsHighValue` | `TotalPrice` > 95th percentile | Flags unusually large orders |
| `PricePerUnit` | `TotalPrice / Quantity` | Normalizes price by quantity ordered |
| `HasDiscount` | `CouponUsed` (from Project 1) | Coupon usage as a fraud-adjacent signal |
| `WeekendFraud` | `IsWeekendOrder × IsFraud` | **Exploratory only** — see next cell |

`WeekendFraud` is a direct function of the label itself (it's literally
`IsWeekendOrder * IsFraud`), so it is used only for the exploratory chart below and
is explicitly excluded from the model's feature matrix — including it as a model
input would be textbook data leakage.

In [4]:
df_features = engineer_fraud_features(df_labeled)
new_cols = [c for c in df_features.columns if c not in df_raw.columns]
print("New columns:", new_cols)
df_features[new_cols].head()

New columns: ['IsFraud', 'AvgItemValue', 'ItemsPerOrder', 'IsHighValue', 'PricePerUnit', 'HasDiscount', 'WeekendFraud']


,IsFraud,AvgItemValue,ItemsPerOrder,IsHighValue,PricePerUnit,HasDiscount,WeekendFraud
0,0,407.586,1.167,1,570.62,1,0
1,0,100.900,1.000,0,151.35,1,0
2,1,344.175,1.333,1,550.68,1,0
3,1,54.638,2.500,0,273.19,1,1
4,0,313.005,1.600,0,626.01,1,0


### 2.1 Exploratory use of `WeekendFraud` (not a model feature)

In [5]:
weekend_fraud_rate = df_features.groupby("IsWeekendOrder")["IsFraud"].mean() * 100
weekend_fraud_rate.index = ["Weekday", "Weekend"]
fig, ax = plt.subplots(figsize=(5, 4))
weekend_fraud_rate.plot(kind="bar", ax=ax, color="#55A868")
ax.set_ylabel("Fraud-proxy rate (%)")
ax.set_title("Fraud-proxy rate: Weekday vs. Weekend orders")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
weekend_fraud_rate

Weekday    40.142518
Weekend    44.413408
Name: IsFraud, dtype: float64

## 3. Assemble the Leak-Free Feature Matrix

Before building `X`, every leaky or non-generalizable column is dropped:

- **`OrderStatus`** — this is the literal source of the label. Leaving it in would let
  the model "predict" fraud by reading the answer key.
- **`WeekendFraud`** — a direct function of the label (see above).
- **Identifiers** — `OrderID`, `CustomerID`, `TrackingNumber`, `ShippingAddress`: no
  generalizable signal, just unique-per-row noise.
- **Already-encoded originals** — raw `Date` (superseded by `OrderMonth` /
  `OrderDayOfWeek` / `IsWeekendOrder`) and raw `CouponCode` (superseded by
  `CouponUsed` / `HasDiscount`).

This is enforced in code by `src/feature_engineering.drop_leaky_columns()`, and
tested explicitly in `tests/test_pipeline.py` (`test_order_status_never_enters_feature_matrix`
and friends) so this guarantee can't silently rot as the project evolves.

In [6]:
X, y = build_feature_matrix(df_features)
print(f"Feature matrix shape: {X.shape}")
assert "OrderStatus" not in X.columns
assert "WeekendFraud" not in X.columns
print("Leakage guard confirmed: neither OrderStatus nor WeekendFraud is in X.")
X.head()

Feature matrix shape: (1200, 37)
Leakage guard confirmed: neither OrderStatus nor WeekendFraud is in X.


,Quantity,UnitPrice,ItemsInCart,TotalPrice,OrderMonth,OrderDayOfWeek,IsWeekendOrder,CouponUsed,CartFillRatio,CustomerOrderCount,IsRepeatCustomer,Quantity_was_outlier,UnitPrice_was_outlier,ItemsInCart_was_outlier,TotalPrice_was_outlier,AvgItemValue,ItemsPerOrder,IsHighValue,PricePerUnit,HasDiscount,Product_Chair,Product_Desk,Product_Laptop,Product_Monitor,Product_Phone,Product_Printer,Product_Tablet,PaymentMethod_Cash,PaymentMethod_Credit Card,PaymentMethod_Debit Card,PaymentMethod_Gift Card,PaymentMethod_Online,ReferralSource_Email,ReferralSource_Facebook,ReferralSource_Google,ReferralSource_Instagram,ReferralSource_Referral
0,5.0,570.62,7.0,2853.10,1.0,2.0,0.0,1.0,0.714,1.0,0.0,0.0,0.0,0.0,0.0,407.586,1.167,1.0,570.62,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,2.0,151.35,3.0,302.70,8.0,4.0,0.0,1.0,0.667,1.0,0.0,0.0,0.0,0.0,0.0,100.900,1.000,0.0,151.35,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,5.0,550.68,8.0,2753.40,2.0,1.0,0.0,1.0,0.625,1.0,0.0,0.0,0.0,0.0,0.0,344.175,1.333,1.0,550.68,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,1.0,273.19,5.0,273.19,10.0,6.0,1.0,1.0,0.200,1.0,0.0,0.0,0.0,0.0,0.0,54.638,2.500,0.0,273.19,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,4.0,626.01,8.0,2504.04,5.0,3.0,0.0,1.0,0.500,1.0,0.0,0.0,0.0,0.0,0.0,313.005,1.600,0.0,626.01,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0


## 4. Stratified Train/Test Split — BEFORE Any Resampling

This is the single most important step in the whole pipeline. The 80/20 split
happens **before** SMOTE or scaling touch the data, and it's **stratified** so the
test set preserves the true class ratio. If SMOTE were applied first and *then*
split, synthetic fraud rows generated from information across the whole dataset
could leak into the test set — inflating every metric with numbers that wouldn't
hold up on genuinely new data.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape[0]} rows ({y_train.mean()*100:.1f}% fraud-proxy)")
print(f"Test:  {X_test.shape[0]} rows ({y_test.mean()*100:.1f}% fraud-proxy)")

Train: 960 rows (41.5% fraud-proxy)
Test:  240 rows (41.2% fraud-proxy)


Notice the fraud-proxy rate is nearly identical between train and test — that's
the stratified split doing its job. The test set's class balance reflects reality;
only the *training* data will be rebalanced with SMOTE next.

## 5. Leak-Free SMOTE Pipelines (imblearn, not sklearn)

A plain `sklearn.pipeline.Pipeline` only knows how to transform `X` — it has no
concept of also changing `y`, which is exactly what SMOTE needs to do (it invents
new minority-class rows with an implied label). `imblearn.pipeline.Pipeline`
understands `fit_resample()`, and critically, only ever applies it to whatever data
it's given at `.fit()` time — which, inside `GridSearchCV`'s cross-validation, is
only the training fold of each split. The validation fold in every CV round stays
untouched by resampling automatically.

- **Logistic Regression pipeline:** `StandardScaler → SMOTE → LogisticRegression`
  (scaling matters here: unscaled features distort the regularization penalty).
- **Random Forest pipeline:** `SMOTE → RandomForestClassifier` (no scaler needed —
  trees split on ordinal thresholds, so scale is irrelevant).

Both are tuned with `GridSearchCV` optimizing for **recall**, not accuracy — in
fraud detection, missing a fraud case (a false negative) is typically far more
costly than a false alarm.

In [8]:
lr_tuned = tune_model(
    build_logistic_regression_pipeline(), LOGISTIC_REGRESSION_PARAM_GRID,
    X_train, y_train, name="Logistic Regression",
)
print("Logistic Regression best params:", lr_tuned.best_params)
print(f"Best CV recall: {lr_tuned.best_cv_recall:.3f}")

Logistic Regression best params: {'classifier__C': 0.1, 'classifier__solver': 'liblinear', 'smote__k_neighbors': 7}
Best CV recall: 0.507


In [9]:
rf_tuned = tune_model(
    build_random_forest_pipeline(), RANDOM_FOREST_PARAM_GRID,
    X_train, y_train, name="Random Forest",
)
print("Random Forest best params:", rf_tuned.best_params)
print(f"Best CV recall: {rf_tuned.best_cv_recall:.3f}")

Random Forest best params: {'classifier__class_weight': None, 'classifier__max_depth': None, 'classifier__n_estimators': 100, 'smote__k_neighbors': 3}
Best CV recall: 0.246


## 6. Evaluate on the Untouched Test Set

**Accuracy is deliberately never computed.** On an imbalanced target, a model
that always predicts "legitimate" would score deceptively well on accuracy while
catching zero fraud — exactly the trap this brief warns about. Precision, Recall,
F1, and ROC-AUC are used instead.

In [10]:
lr_result = evaluate_model(lr_tuned.best_estimator, X_test, y_test, "Logistic Regression")
rf_result = evaluate_model(rf_tuned.best_estimator, X_test, y_test, "Random Forest")
results = [lr_result, rf_result]

metrics_df = results_to_dataframe(results)
metrics_df

,model,precision,recall,f1,roc_auc
0,Logistic Regression,0.377,0.434,0.404,0.437
1,Random Forest,0.368,0.253,0.299,0.469


In [11]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
plot_confusion_matrix(axes[0, 0], lr_result)
plot_confusion_matrix(axes[0, 1], rf_result)
plot_roc_curves(axes[0, 2], results)
plot_precision_recall_curves(axes[1, 0], results)
plot_feature_importance(axes[1, 1], rf_result)
plot_metric_comparison(axes[1, 2], results)
fig.suptitle("Fraud Detection Dashboard", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

## 7. Interpreting the Results Honestly

The ROC-AUC for both models lands close to **0.45–0.50** — essentially indistinguishable
from a random classifier (the diagonal dashed line on the ROC plot). This is a real,
important finding, not a bug in the pipeline:

- The synthetic Project 1 dataset assigns `OrderStatus` with no genuine causal link to
  any of the order's other attributes (price, product, timing, etc.) — the generator
  appears to pick statuses close to uniformly at random.
- That means the proxy label `IsFraud` (Returned/Cancelled) has **no real signal** for
  these models to learn from the features available. The pipeline is doing its job
  correctly; there's simply nothing predictive to find in this particular dataset.
- This is exactly why the leak-free discipline in this notebook matters: with a
  leaky pipeline (SMOTE before splitting, or `OrderStatus` in the features), it would
  have been easy to report a deceptively good-looking score that doesn't reflect
  reality. The honest, near-random result here is actually evidence the pipeline is
  trustworthy.
- On real transaction data with genuine behavioral fraud signals, this same
  architecture (leak-free split → SMOTE-in-training-fold → recall-optimized
  GridSearchCV → Precision/Recall/ROC-AUC evaluation) is exactly what you'd want —
  it would simply have real signal to find.

**Model comparison, on its own terms:** whichever model comes out ahead on recall in
`reports/model_evaluation_metrics.csv` is the "better" pick by the brief's own
criterion (recall-first, because missing fraud is costlier than a false alarm) — but
given the near-random ROC-AUC, neither model should be read as a meaningfully
predictive fraud detector on this particular dataset.

## 8. Conclusion

This project delivers a complete, leak-free supervised classification pipeline:

- A clearly-labeled proxy fraud target, with the simplification stated up front
  rather than implied.
- 6 engineered features, one deliberately excluded from modeling as a
  leakage guard, tested in code.
- An 80/20 stratified split performed **before** any resampling.
- Two models (Logistic Regression, Random Forest) tuned via `GridSearchCV` on
  **recall**, with SMOTE safely isolated inside `imblearn.pipeline.Pipeline` so it
  never touches validation or test folds.
- Evaluation via Precision/Recall/F1/ROC-AUC — accuracy deliberately excluded.
- An honest read of the results: near-random ROC-AUC reported plainly, with the
  reason explained, rather than a misleadingly polished number.

Outputs:
- `data/processed/orders_with_fraud_features.csv` — full feature-engineered dataset
- `reports/model_evaluation_metrics.csv`, `reports/*_best_params.csv`,
  `reports/random_forest_feature_importance.csv`
- `reports/figures/00_class_balance.png`, `reports/figures/01_fraud_detection_dashboard.png`